In [51]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
df = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 


In [52]:
statlog_german_credit_data["metadata"]["additional_info"]["variable_info"]

'Attribute 1:  (qualitative)      \r\n Status of existing checking account\r\n             A11 :      ... <    0 DM\r\n\t       A12 : 0 <= ... <  200 DM\r\n\t       A13 :      ... >= 200 DM / salary assignments for at least 1 year\r\n               A14 : no checking account\r\n\r\nAttribute 2:  (numerical)\r\n\t      Duration in month\r\n\r\nAttribute 3:  (qualitative)\r\n\t      Credit history\r\n\t      A30 : no credits taken/ all credits paid back duly\r\n              A31 : all credits at this bank paid back duly\r\n\t      A32 : existing credits paid back duly till now\r\n              A33 : delay in paying off in the past\r\n\t      A34 : critical account/  other credits existing (not at this bank)\r\n\r\nAttribute 4:  (qualitative)\r\n\t      Purpose\r\n\t      A40 : car (new)\r\n\t      A41 : car (used)\r\n\t      A42 : furniture/equipment\r\n\t      A43 : radio/television\r\n\t      A44 : domestic appliances\r\n\t      A45 : repairs\r\n\t      A46 : education\r\n\t      A47 : 

In [53]:
df.columns

Index(['Attribute1', 'Attribute2', 'Attribute3', 'Attribute4', 'Attribute5',
       'Attribute6', 'Attribute7', 'Attribute8', 'Attribute9', 'Attribute10',
       'Attribute11', 'Attribute12', 'Attribute13', 'Attribute14',
       'Attribute15', 'Attribute16', 'Attribute17', 'Attribute18',
       'Attribute19', 'Attribute20'],
      dtype='object')

In [54]:
statlog_german_credit_data.variables

variables = ["status_account", "duration", "credit_history", "purpose", "obligo", 
             "savings", "employed_since", "installment", "marital_status", "debtors",
             "residence_since", "property", "age", "other_installment_plans", "housing", 
             "n_credits", "job", "people_liable", "telephone", "foreign_worker"]

df.columns = variables

df["target"] = y - 1

In [55]:
categorical_cols = df.select_dtypes(include=['object']).columns

# 2. Convert those specific columns into one-hot encoded variables
# We use columns=categorical_cols so it leaves your numerical data untouched
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.head())

   duration  obligo  installment  residence_since  age  n_credits  \
0         6    1169            4                4   67          2   
1        48    5951            2                2   22          1   
2        12    2096            2                3   49          1   
3        42    7882            2                4   45          1   
4        24    4870            3                4   53          2   

   people_liable  target  status_account_A12  status_account_A13  ...  \
0              1       0               False               False  ...   
1              1       1                True               False  ...   
2              2       0               False               False  ...   
3              2       0               False               False  ...   
4              2       1               False               False  ...   

   property_A124  other_installment_plans_A142  other_installment_plans_A143  \
0          False                         False                    

## Logisitic Regression

In [65]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report 
lr_model = LogisticRegression(max_iter=300, class_weight="balanced")
train, test = train_test_split(df_encoded, test_size=0.3, random_state=42)
train_y = train.target
test_y = test.target

train_features = train.drop(columns="target")
test_features = test.drop(columns="target")

# Find out which boolean columns might significantly distinguish between good and bad credit
sig_cols = []

for col in train_features.columns: 
    if train[col].dtype == bool:

        # Create a cross-tabulation (contingency table)
        contingency_table = pd.crosstab(train["target"], train[col])

        chi2, p, dof, expected = chi2_contingency(contingency_table)
        if p < 0.05:
            print(f"{col}: P-value: {p:.5f}") # If p < 0.05, this model has predictive power
            sig_cols.append(col)

sig_cols.extend(train_features.select_dtypes(include=['int64']).columns)
print(sig_cols)

scaler = StandardScaler()
numeric_features = scaler.fit_transform(train_features[["obligo", "duration", "age"]])
numeric_features_test = scaler.transform(test_features[["obligo", "duration", "age"]])

# Train logistic regression model 
train_filtered = train_features[sig_cols].copy()
train_filtered[["obligo", "duration", "age"]] = numeric_features
lr_model.fit(train_filtered, y=train_y)

test_filtered = test_features[sig_cols].copy()
test_filtered[["obligo", "duration", "age"]] = numeric_features_test
test_predictions = lr_model.predict(test_filtered)

print(classification_report(y_true=test_y, y_pred=test_predictions))

status_account_A12: P-value: 0.00219
status_account_A14: P-value: 0.00000
credit_history_A31: P-value: 0.00134
credit_history_A32: P-value: 0.04910
credit_history_A34: P-value: 0.00000
purpose_A41: P-value: 0.02498
purpose_A43: P-value: 0.00061
purpose_A46: P-value: 0.02380
savings_A63: P-value: 0.03109
savings_A64: P-value: 0.02415
savings_A65: P-value: 0.03288
employed_since_A72: P-value: 0.02837
employed_since_A75: P-value: 0.02502
marital_status_A92: P-value: 0.01899
marital_status_A93: P-value: 0.02556
property_A124: P-value: 0.00036
other_installment_plans_A143: P-value: 0.02816
housing_A152: P-value: 0.00009
housing_A153: P-value: 0.00411
['status_account_A12', 'status_account_A14', 'credit_history_A31', 'credit_history_A32', 'credit_history_A34', 'purpose_A41', 'purpose_A43', 'purpose_A46', 'savings_A63', 'savings_A64', 'savings_A65', 'employed_since_A72', 'employed_since_A75', 'marital_status_A92', 'marital_status_A93', 'property_A124', 'other_installment_plans_A143', 'housing

### Next steps
- Feature Engineering (age, duration, obligo)
- regularization 
- adjusting the decision threshold 
- checking multicollinearity 